# Current Fuzzy Logic Controller Architecture

This document describes the current architecture of the fuzzy controller used for the Asteroids project. The goal is to separate the game simulation from the controller so that the controller behaves like any other control system (PID, LQR, Dynamic Inversion, etc.).

---

# High-Level Architecture

```text
                  +----------------------+
                  |      Game Engine     |
                  |----------------------|
                  | Physics              |
                  | Rendering            |
                  | Collision Detection  |
                  +----------+-----------+
                             |
                             | Game State
                             ▼
                  +----------------------+
                  |   Fuzzy Controller   |
                  |----------------------|
                  | Fuzzification        |
                  | Rule Evaluation      |
                  | Aggregation          |
                  | Defuzzification      |
                  +----------+-----------+
                             |
                             | Control Commands
                             ▼
                  +----------------------+
                  |      Game Engine     |
                  |----------------------|
                  | Apply Controls       |
                  | Update Physics       |
                  +----------------------+
```

The controller never directly manipulates the game. Instead:

1. The game exposes its current state.
2. The controller computes control commands.
3. The game applies those commands.

---

# Separation of Responsibilities

## Game

Responsible for:

- Physics
- Rendering
- Collision detection
- Target spawning
- Entity management

The game should expose a simplified state through something like:

```python
game.get_state()
```

Example:

```python
{
    "heading_error": ...,
    "distance_error": ...,
    "speed": ...
}
```

The game should also accept controller outputs:

```python
game.apply_control(control)
```

Example:

```python
{
    "turn": ...,
    "thrust": ...
}
```

---

## Controller

Responsible only for mapping:

```text
State
    ↓
Control
```

It knows nothing about:

- sprites
- rendering
- pygame
- collision detection

Its only job is:

```python
control = controller.compute_control(state)
```

---

# Controller Pipeline

```text
Crisp Inputs
      │
      ▼
Fuzzification
      │
      ▼
Rule Evaluation
      │
      ▼
Aggregation
      │
      ▼
Defuzzification
      │
      ▼
Control Outputs
```

---

# Step 1 — Crisp Inputs

Initially the controller uses:

```text
heading_error
distance_error
speed
```

Possible future inputs:

- heading rate
- closing velocity
- angular velocity
- obstacle distance
- obstacle heading

These values are ordinary floating point numbers.

Example:

```python
heading_error = -25°
distance_error = 350 px
speed = 1.8 px/frame
```

---

# Step 2 — Membership Functions

Each input is represented by overlapping fuzzy sets.

Example:

```text
Heading Error

Hard Left
      Left
          Center
                Right
                      Hard Right
```

Each set is represented by a membership function.

Current implementation uses triangular membership functions.

Example:

```text
        1
       /\
      /  \
_____/____\_____
```

Each triangle is defined by

```python
[left, center, right]
```

Example:

```python
[-180, -90, 0]
```

---

# Step 3 — Fuzzification

Each crisp input is evaluated against every membership function.

Example:

```python
heading_error = -30°
```

might produce

```python
{
    "left": 0.25,
    "center": 0.75,
    "right": 0.0
}
```

This means

> The heading error is simultaneously 25% Left and 75% Center.

---

# Step 4 — Rule Base

The rule base stores expert knowledge.

Example:

```text
IF Heading is Left
THEN Turn Left
```

or

```text
IF Heading is Center
AND Distance is Far
THEN High Thrust
```

Rules are independent of the game implementation.

---

# Step 5 — Rule Evaluation

Each rule computes an activation value.

For an AND rule:

```text
IF Heading = Left
AND Distance = Far
```

activation is

```text
min(
    Heading_Left,
    Distance_Far
)
```

Example:

```python
Heading_Left = 0.6
Distance_Far = 0.8
```

activation

```python
0.6
```

---

# Step 6 — Aggregation

Many rules may recommend the same output.

Example:

```text
Turn Left

Rule 1 → 0.7

Rule 2 → 0.3

Rule 3 → 0.5
```

These are combined before producing the final output.

---

# Step 7 — Defuzzification

The aggregated fuzzy outputs become crisp control commands.

Current plan:

Use weighted average (Sugeno-style).

Produces

```python
turn = -0.42
thrust = 0.81
```

which the game can directly apply.

---

# Controller Class Layout

```text
FuzzyController
│
├── Input Membership Functions
│
├── Output Membership Functions
│
├── Rule Base
│
├── Fuzzification
│
├── Rule Evaluation
│
├── Aggregation
│
└── Defuzzification
```

The main interface becomes

```python
compute_control(state)
```

which returns

```python
{
    "turn": ...,
    "thrust": ...
}
```

---

# Project Architecture

```text
project/

game/
│
├── game.py
├── physics.py
├── entities.py
└── rendering.py

control/
│
├── fuzzy_controller.py
├── membership.py
├── rules.py
├── inference.py
├── defuzzification.py
└── utils.py

training/
│
├── chromosome.py
├── ga.py
└── cost.py

main.py
```

---

# Planned Development Order

## Phase 1

Expose the game state.

```python
state = game.get_state()
```

---

## Phase 2

Implement only

```text
heading_error
        ↓
turn
```

Ignore thrust completely.

This allows turning behavior to be debugged independently.

---

## Phase 3

Add

```text
distance_error
        ↓
thrust
```

using a simple rule set.

---

## Phase 4

Include

```text
speed
```

to allow the controller to regulate approach velocity.

---

## Phase 5

Introduce the genetic algorithm.

Initially evolve:

- membership function locations
- rule table

Later evolve:

- output singleton values
- membership widths
- cost function weights

---

# Long-Term Goal

Eventually the controller should look identical to a traditional control system from the outside.

```text
            State
              │
              ▼
      Fuzzy Controller
              │
              ▼
        Control Vector
```

Mathematically,

```text
u = F(x)
```

where

- x = game state
- u = control vector
- F = fuzzy inference system

From the game's perspective, it should not matter whether `F` is implemented using:

- PID
- LQR
- Dynamic Inversion
- MPC
- Fuzzy Logic

The game simply provides the current state and applies the returned control commands.